In [1]:
from anarcii import Anarcii # https://github.com/oxpig/ANARCII (! pip install anarcii)
import pandas as pd
import numpy as np
import os 


raw_datasets = pd.read_csv(os.path.join("datasets", "cleaned_seqs_all_seq_1k.csv"), index_col=0)  # Cleaned sequences dataset

FileNotFoundError: [Errno 2] No such file or directory: 'datasets\\cleaned_seqs_all_seq_1k.csv'

> Steps required:
1. Pre-process raw sequence.
2. Translate raw sequence.
3. Apply the Anarchii model on the sequence.
4. Compare results with old data.

In [ ]:
################################################################
codon_dict = {   'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
                 'TCT': "S", 'TCC': "S", 'TCA': "S", 'TCG': "S",
                 'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',  # * for STOP
                 'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',

                 'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
                 'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
                 'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
                 'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',

                 'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
                 'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
                 'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
                 'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',

                 'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
                 'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
                 'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
                 'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
                }


################################
def use_anarchii(list_seqs: list,
                 model:str = "antibody"):
    """
    Helper function that use to anarcii model to number amino acids positions according to the IMGT numbering scheme

    list_seq: array-like -> list of AA sequence to number.
    model: str -> Which model of the anarcii algorithm to use (see doc for other options).
    """
    
    # Select the type of sequence (antibody, tcr, shark or unknown) and instantiate the model. 
    model = Anarcii(seq_type="antibody")

    # Call the number method on a list of sequences, path to a fasta or PDB file.
    results = model.number(list_seqs)

    return results


##############################
def translate(seq:str,
              nt_start:int = 1,
              nt_end:int = None) -> str:
    """
    Helper function that translate NT DNA sequence to AA sequence.

    seq:str -> string of AA sequence to be translated.
    nt_start:int -> Position of NT from which the translation will begin.
    nt_end:int ->  Position of NT in which the translation will end.
    """

    # Getting final NT for translation
    if isinstance(nt_end, int) & nt_end <= len(seq):
        nt_stop = nt_end
    else:
        nt_stop = len(seq)

    seq_range = range(nt_start, nt_stop)
    translated = []

    # Translating sequence
    for i in seq_range:
        codon = seq[i*3-3:i*3]

        if codon in list(codon_dict.keys()):
            aa = codon_dict[codon] 

        elif codon == "---":
            aa = "-"
            
        else:
            aa = "X"
         
        translated.append(aa)

    return "".join(translated)



######################################
def process_sequence(seq: str,
                     anarcii: bool = True) -> str:
    """
    Custom function that process our ImmuneDB sequence for the processing of the Anarcii algorighm.
    (Should work on any NT DNA sequence, it's just won't utilize all of the steps.)
    Steps:
   
    seq: str -> DNA NT sequence in string format.
    anarcii: bool -> if to use the anarcii algorithm so assign IMGT numbering.
    """
    # Counting spcaers ("-") and sequencing unknown AAs ("N")
    n_spacers, n_seqerror = seq.count("-"), seq.count("N")
    div3_spacers, div3_seqerror = seq.count("-") % 3, seq.count("N") % 3

    # If number of spacers isn't dividing by 3 -> raise an error.
    if div3_spacers != 0:
        return np.nan

    # Removing spacers
    if anarcii:
        seq2translate = seq.replace("-","")
    else: 
        seq2translate = seq

    seq_length = len(seq2translate)
    seq_rem = seq_length % 3

    # making sure the sequence is divided by 3 before translation
    if seq_rem != 0:
        seq_range = range(1,seq_length - seq_rem)
    else:
        seq_range = range(1,seq_length)




    if anarcii:
        seq_aa = "".join(translated).replace("N","")
        seq_aa_anarchii = use_anarchii(seq_aa)
        seq_result = "".join([i[1] for i in seq_aa_anarchii["Sequence"]["numbering"]])

    else:
        seq_result = "".join(translated)
        #seq_result = seq_result.replace("N","-")


    return seq_result

In [ ]:
demo_seq = raw_datasets.sequence[0]
aa_anarcii = process_sequence(demo_seq, anarcii=True)
aa_og = process_sequence(seq=demo_seq, anarcii=False)

In [ ]:
aa_anarcii

In [ ]:
aa_og

In [ ]:
raw_datasets.iloc[:1000,:].to_csv(os.path.join("data", "cleaned_seqs_all_seq_1k.csv"))